# Phase 7 — Hyperparameter Tuning with Keras Tuner

**Goal:** Systematically search for the best neural network architecture and training settings using automated hyperparameter search. Compare RandomSearch and Hyperband strategies.

**Libraries:** `keras-tuner`, `tensorflow`, `pandas`, `scikit-learn`, `matplotlib`

**Input:** `../star_dataset_clean.csv`

**Install if needed:** `pip install keras-tuner`

In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import tensorflow as tf
import keras_tuner as kt

os.makedirs("../figures", exist_ok=True)
print("TensorFlow:", tf.__version__, " | keras-tuner:", kt.__version__)

TensorFlow: 2.21.0  | keras-tuner: 1.4.8


In [6]:
df = pd.read_csv("../star_dataset_clean.csv")

feature_cols = ['Distance (ly)', 'Luminosity (L/Lo)', 'Radius (R/Ro)', 'Temperature (K)']
X = df[feature_cols].values
y = df['Spectral Class'].values

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
n_classes = len(label_encoder.classes_)

# Same 70 / 15 / 15 split as phase 6
X_temp, X_test, y_temp, y_test = train_test_split(X, y_encoded, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.18, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}  Val: {X_val_scaled.shape}  Test: {X_test_scaled.shape}")
print(f"Classes: {n_classes}")

Train: (19, 4)  Val: (5, 4)  Test: (5, 4)
Classes: 28


In [7]:
def build_model(hp):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(4,)),
        tf.keras.layers.Dense(
            units=hp.Int('units_1', min_value=32, max_value=128, step=32),
            activation='relu'
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(
            rate=hp.Float('dropout', min_value=0.2, max_value=0.5, step=0.1)
        ),
        tf.keras.layers.Dense(
            units=hp.Int('units_2', min_value=16, max_value=64, step=16),
            activation='relu'
        ),
        tf.keras.layers.Dense(n_classes, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("build_model defined — hyperparameters: units_1, units_2, dropout, learning_rate")

build_model defined — hyperparameters: units_1, units_2, dropout, learning_rate


In [8]:
tuner_rs = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='tuner_logs',
    project_name='random_search',
    overwrite=True,
)

tuner_rs.search_space_summary()

tuner_rs.search(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)],
    verbose=0,
)

print("\n=== RandomSearch Results ===")
tuner_rs.results_summary()

best_hp_rs = tuner_rs.get_best_hyperparameters(num_trials=1)[0]
print("\nBest hyperparameters (RandomSearch):")
print(best_hp_rs.values)

Search space summary
Default search space size: 4
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 128, 'step': 32, 'sampling': 'linear'}
dropout (Float)
{'default': 0.2, 'conditions': [], 'min_value': 0.2, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}
units_2 (Int)
{'default': None, 'conditions': [], 'min_value': 16, 'max_value': 64, 'step': 16, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


Traceback (most recent call last):
  File "c:\Users\ajgeb\Desktop\Tech & Coding\Coding_Project\astronomy_data_analysis\.venv\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ajgeb\Desktop\Tech & Coding\Coding_Project\astronomy_data_analysis\.venv\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
  File "c:\Users\ajgeb\Desktop\Tech & Coding\Coding_Project\astronomy_data_analysis\.venv\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 309, in run_trial
    self._configure_tensorboard_dir(callbacks, trial, execution)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ajgeb\Desktop\Tech & Coding\Coding_Project\astronomy_data_analysis\.venv\Lib\site-packages\keras_t

RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "c:\Users\ajgeb\Desktop\Tech & Coding\Coding_Project\astronomy_data_analysis\.venv\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ajgeb\Desktop\Tech & Coding\Coding_Project\astronomy_data_analysis\.venv\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
  File "c:\Users\ajgeb\Desktop\Tech & Coding\Coding_Project\astronomy_data_analysis\.venv\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 309, in run_trial
    self._configure_tensorboard_dir(callbacks, trial, execution)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ajgeb\Desktop\Tech & Coding\

In [ ]:
tuner_hb = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=50,
    factor=3,
    directory='tuner_logs',
    project_name='hyperband',
    overwrite=True,
)

tuner_hb.search(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)],
    verbose=0,
)

print("\n=== Hyperband Results ===")
tuner_hb.results_summary()

best_hp_hb = tuner_hb.get_best_hyperparameters(num_trials=1)[0]
print("\nBest hyperparameters (Hyperband):")
print(best_hp_hb.values)

# Compare best val_accuracy between the two strategies
rs_best_score = tuner_rs.oracle.get_best_trials(num_trials=1)[0].score
hb_best_score = tuner_hb.oracle.get_best_trials(num_trials=1)[0].score
print(f"\nRandomSearch best val_accuracy: {rs_best_score:.4f}")
print(f"Hyperband    best val_accuracy: {hb_best_score:.4f}")

In [ ]:
# Combine train + val — tuning is done, no longer need a held-out val set
X_trainval = np.concatenate([X_train_scaled, X_val_scaled])
y_trainval  = np.concatenate([y_train, y_val])

best_model = build_model(best_hp_rs)
best_model.fit(X_trainval, y_trainval, epochs=50, batch_size=8, verbose=0)

test_loss, test_acc = best_model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Tuned model — Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.4f}")

y_pred = np.argmax(best_model.predict(X_test_scaled, verbose=0), axis=1)
present_labels = sorted(set(y_test) | set(y_pred))
present_names  = label_encoder.inverse_transform(present_labels)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_names))

In [ ]:
# Collect val_accuracy score for every RandomSearch trial
trial_scores = []
for trial_id, trial in tuner_rs.oracle.trials.items():
    if trial.score is not None:
        trial_scores.append((trial_id, trial.score))

trial_scores.sort(key=lambda x: x[0])   # sort by trial id
ids    = [t[0] for t in trial_scores]
scores = [t[1] for t in trial_scores]

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(scores) + 1), scores, 'o-', color='steelblue', linewidth=1.5)
plt.axhline(max(scores), color='red', linestyle='--', linewidth=1, label=f"Best: {max(scores):.4f}")
plt.xlabel("Trial")
plt.ylabel("Val Accuracy")
plt.title("RandomSearch: Val Accuracy per Trial")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/phase7_search_landscape.png", dpi=150)
plt.show()